# 04 — Seleção da amostra e conectividade funcional

Este notebook realiza a seleção da amostra final a partir dos dados de rs-fMRI processados pelo pipeline Athena e do atlas CC200.

Etapas:

1. Localizar os arquivos `sfnwmrda*_session_1_rest_1_cc200_TCs.1D`;
2. Identificar os participantes disponíveis nos dados de rs-fMRI;
3. Cruzar os participantes com o arquivo fenotípico do ADHD-200;
4. Aplicar `QC_Athena == 1`;
5. Excluir diagnósticos pendentes;
6. Transformar os diagnósticos em classificação binária:
   - `0` = Controle;
   - `1` = TDAH;
7. Verificar a distribuição das classes;
8. Calcular a matriz de conectividade funcional por correlação de Pearson;
9. Verificar a presença das 190 ROIs do atlas CC200;
10. Salvar as matrizes de conectividade e a tabela final de participantes.

> A amostra final será determinada pela interseção entre os participantes com diagnóstico válido, aprovação no QC_Athena e disponibilidade dos arquivos de rs-fMRI CC200.

In [8]:
# Configuração e carregamento dos dados


from pathlib import Path
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from nilearn.connectome import ConnectivityMeasure


# Localizar raiz do projeto


def encontrar_raiz_projeto():
    candidatos = [Path.cwd(), *Path.cwd().parents]

    for caminho in candidatos:
        if (caminho / "config.py").exists():
            return caminho.resolve()

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto TCC_TDAH."
    )


RAIZ = encontrar_raiz_projeto()

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import config

config.criar_pastas()


# Verificar arquivo fenotípico


if not config.ARQ_PHENO.exists():
    raise FileNotFoundError(
        f"Arquivo de fenótipos não encontrado:\n"
        f"{config.ARQ_PHENO}"
    )

pheno = pd.read_csv(
    config.ARQ_PHENO,
    sep="\t"
)


# Padronizar DX


pheno["DX"] = pd.to_numeric(
    pheno["DX"],
    errors="coerce"
)


# Padronizar identificador


if "ScanDir ID" not in pheno.columns:
    raise KeyError(
        "A coluna 'ScanDir ID' não foi encontrada no "
        "arquivo fenotípico."
    )

pheno["ID_padronizado"] = (
    pd.to_numeric(
        pheno["ScanDir ID"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
)

print("=" * 60)
print("CONFIGURAÇÃO — NOTEBOOK 04")
print("=" * 60)

print(f"\nRaiz do projeto:")
print(f"  {RAIZ}")

print(f"\nDados extraídos:")
print(f"  {config.EXTRACTED}")

print(f"\nConectividade:")
print(f"  {config.CONNECTIVITY}")

print(f"\nFenótipo:")
print(f"  {config.ARQ_PHENO}")

print(f"\nParticipantes no fenótipo: {len(pheno):,}")

CONFIGURAÇÃO — NOTEBOOK 04

Raiz do projeto:
  /home/maria/TCC_TDAH

Dados extraídos:
  /home/maria/TCC_TDAH/data/extracted

Conectividade:
  /home/maria/TCC_TDAH/data/connectivity

Fenótipo:
  /home/maria/TCC_TDAH/data/raw/adhd200_preprocessed_phenotypics.tsv

Participantes no fenótipo: 973


In [9]:
# Localizar arquivos CC200

registros = []

padrao = "sfnwmrda*_session_1_rest_1_cc200_TCs.1D"

for p in config.EXTRACTED.rglob(padrao):

    registros.append({
        "site": p.parents[1].name,
        "subject_id": p.parent.name,
        "arquivo": p.name,
        "caminho": str(p)
    })

df_exames = pd.DataFrame(registros)

if df_exames.empty:
    raise FileNotFoundError(
        "Nenhum arquivo CC200 foi encontrado.\n\n"
        f"Pasta pesquisada:\n{config.EXTRACTED}\n\n"
        f"Padrão procurado:\n{padrao}"
    )


# Padronizar ID

df_exames["ID_padronizado"] = (
    pd.to_numeric(
        df_exames["subject_id"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
)

# Remover IDs inválidos
df_exames = df_exames[
    df_exames["ID_padronizado"] != "<NA>"
].copy()

print("=" * 60)
print("ARQUIVOS CC200 ENCONTRADOS")
print("=" * 60)

print(
    f"\nArquivos CC200 filtrados: "
    f"{len(df_exames):,}"
)

print(
    f"IDs únicos: "
    f"{df_exames['ID_padronizado'].nunique():,}"
)

print("\nParticipantes por site:")

display(
    df_exames["site"]
    .value_counts()
    .sort_index()
)

print("\nPrimeiros registros:")

display(
    df_exames.head()
)

ARQUIVOS CC200 ENCONTRADOS

Arquivos CC200 filtrados: 385
IDs únicos: 385

Participantes por site:


site
KKI            83
NYU           216
NeuroIMAGE     48
OHSU           38
Name: count, dtype: int64


Primeiros registros:


,site,subject_id,arquivo,caminho,ID_padronizado
0,NeuroIMAGE,1208586,sfnwmrda1208586_session_1_rest_1_cc200_TCs.1D,/home/maria/TCC_TDAH/data/extracted/ADHD200_CC...,1208586
1,NeuroIMAGE,2756846,sfnwmrda2756846_session_1_rest_1_cc200_TCs.1D,/home/maria/TCC_TDAH/data/extracted/ADHD200_CC...,2756846
2,NeuroIMAGE,1438162,sfnwmrda1438162_session_1_rest_1_cc200_TCs.1D,/home/maria/TCC_TDAH/data/extracted/ADHD200_CC...,1438162
3,NeuroIMAGE,1585708,sfnwmrda1585708_session_1_rest_1_cc200_TCs.1D,/home/maria/TCC_TDAH/data/extracted/ADHD200_CC...,1585708
4,NeuroIMAGE,3515506,sfnwmrda3515506_session_1_rest_1_cc200_TCs.1D,/home/maria/TCC_TDAH/data/extracted/ADHD200_CC...,3515506


In [10]:
# 3. Cruzar dados de rs-fMRI com o fenótipo

df_part = df_exames.merge(
    pheno,
    on="ID_padronizado",
    how="left",
    suffixes=("_exame", "_pheno"),
    validate="many_to_one"
)

print("=" * 60)
print("CRUZAMENTO — CC200 × FENÓTIPO")
print("=" * 60)

print(f"\nArquivos CC200: {len(df_exames):,}")
print(f"Registros após cruzamento: {len(df_part):,}")

print(
    f"\nSem diagnóstico (DX): "
    f"{df_part['DX'].isna().sum():,}"
)

print(
    f"Sem QC_Athena: "
    f"{df_part['QC_Athena'].isna().sum():,}"
)

print("\nDistribuição do DX:")

display(
    df_part["DX"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDX × QC_Athena:")

display(
    pd.crosstab(
        df_part["DX"],
        df_part["QC_Athena"],
        margins=True,
        dropna=False
    )
)

CRUZAMENTO — CC200 × FENÓTIPO

Arquivos CC200: 385
Registros após cruzamento: 385

Sem diagnóstico (DX): 0
Sem QC_Athena: 0

Distribuição do DX:


DX
0.0    202
1.0    121
2.0      9
3.0     53
Name: count, dtype: int64


DX × QC_Athena:


QC_Athena,0.0,1.0,All
DX,,,
0.0,17,185,202
1.0,12,109,121
2.0,1,8,9
3.0,9,44,53
All,39,346,385


In [11]:
# Seleção da amostra final

df_sel = df_part[
    (df_part["QC_Athena"] == 1) &
    (df_part["DX"].isin([0, 1, 2, 3]))
].copy()


# Classificação binária

df_sel["classe"] = np.where(
    df_sel["DX"] == 0,
    0,
    1
)

print("=" * 60)
print("AMOSTRA SELECIONADA")
print("=" * 60)

print(
    f"\nParticipantes/arquivos selecionados: "
    f"{len(df_sel):,}"
)

print("\nDiagnóstico original:")

display(
    df_sel["DX"]
    .value_counts()
    .sort_index()
)

print("\nClasses finais:")

contagem_classes = (
    df_sel["classe"]
    .value_counts()
    .sort_index()
)

display(contagem_classes)


# Percentuais


percentuais = (
    df_sel["classe"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("\nDistribuição percentual:")

for classe, percentual in percentuais.items():

    nome = "Controle" if classe == 0 else "TDAH"

    quantidade = contagem_classes[classe]

    print(
        f"{nome}: {quantidade:,} "
        f"({percentual:.2f}%)"
    )


# Razão de desbalanceamento


if len(contagem_classes) == 2:

    maior = contagem_classes.max()
    menor = contagem_classes.min()

    print(
        f"\nRazão maior/menor classe: "
        f"{maior / menor:.2f}:1"
    )

AMOSTRA SELECIONADA

Participantes/arquivos selecionados: 346

Diagnóstico original:


DX
0.0    185
1.0    109
2.0      8
3.0     44
Name: count, dtype: int64


Classes finais:


classe
0    185
1    161
Name: count, dtype: int64


Distribuição percentual:
Controle: 185 (53.47%)
TDAH: 161 (46.53%)

Razão maior/menor classe: 1.15:1


In [12]:

# 5. Função para calcular conectividade funcional


def calcular_conectividade(caminho):

    df = pd.read_csv(
        caminho,
        sep="\t"
    )

    # Padronizar nomes das colunas
    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    # Selecionar séries temporais das ROIs


    cols = [
        c for c in df.columns
        if c.startswith("Mean_")
    ]

    if len(cols) != config.N_ROIS:
        raise ValueError(
            f"Número de ROIs encontrado: {len(cols)}. "
            f"Esperado: {config.N_ROIS}."
        )

    ts = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .values
    )


    # Verificar valores inválidos


    if not np.isfinite(ts).all():
        raise ValueError(
            "A série temporal contém NaN ou Inf."
        )


    # Calcular correlação de Pearson


    medida = ConnectivityMeasure(
        kind="correlation"
    )

    fc = medida.fit_transform([ts])[0]


    # Validar matriz


    if fc.shape != (
        config.N_ROIS,
        config.N_ROIS
    ):
        raise ValueError(
            f"Dimensão inesperada da matriz FC: "
            f"{fc.shape}"
        )

    if not np.allclose(
        fc,
        fc.T,
        atol=1e-6
    ):
        raise ValueError(
            "A matriz de conectividade não é simétrica."
        )

    if not np.isfinite(fc).all():
        raise ValueError(
            "A matriz de conectividade contém NaN/Inf."
        )


    # Identificadores das ROIs


    roi_ids = np.array([
        int(c.split("_")[1])
        for c in cols
    ])

    return fc, roi_ids

In [13]:

# 6. Calcular e salvar conectividade


erros = []
processados = 0
roi_ref = None

config.CONNECTIVITY.mkdir(
    parents=True,
    exist_ok=True
)

for _, r in tqdm(
    df_sel.iterrows(),
    total=len(df_sel),
    desc="Calculando conectividade"
):

    try:

        fc, roi_ids = calcular_conectividade(
            r["caminho"]
        )


        # Verificar consistência das ROIs


        if roi_ref is None:

            roi_ref = roi_ids

        elif not np.array_equal(
            roi_ref,
            roi_ids
        ):

            raise ValueError(
                "O conjunto/ordem das ROIs "
                "difere do primeiro participante."
            )


        # Nome do arquivo


        destino = (
            config.CONNECTIVITY /
            f"FC_{r['ID_padronizado']}.npy"
        )

        np.save(
            destino,
            fc
        )

        processados += 1

    except Exception as e:

        erros.append({
            "ID": r["ID_padronizado"],
            "arquivo": r["arquivo"],
            "erro": str(e)
        })


print("\n" + "=" * 60)
print("RESULTADO DO PROCESSAMENTO")
print("=" * 60)

print(
    f"\nProcessados com sucesso: "
    f"{processados:,}"
)

print(
    f"Erros: "
    f"{len(erros):,}"
)

if erros:

    print("\nParticipantes com erro:")

    display(
        pd.DataFrame(erros)
    )

else:

    print("\n✓ Nenhum erro encontrado.")

Calculando conectividade:   0%|          | 0/346 [00:00<?, ?it/s]


RESULTADO DO PROCESSAMENTO

Processados com sucesso: 346
Erros: 0

✓ Nenhum erro encontrado.


In [14]:

# 7. Consolidar a amostra efetivamente processada


ids_ok = {
    p.stem.replace("FC_", "")
    for p in config.CONNECTIVITY.glob("FC_*.npy")
}

df_final = (
    df_sel[
        df_sel["ID_padronizado"].isin(ids_ok)
    ]
    .drop_duplicates(
        subset=["ID_padronizado"]
    )
    .reset_index(drop=True)
)


# Verificar se existe referência de ROIs


if roi_ref is None:

    raise RuntimeError(
        "Nenhuma matriz de conectividade foi "
        "processada com sucesso."
    )


# Salvar participantes


df_final.to_csv(
    config.ARQ_PARTICIPANTES,
    index=False
)


# Salvar IDs das ROIs


np.save(
    config.ARQ_ROIS,
    roi_ref
)

print("=" * 60)
print("AMOSTRA FINAL PROCESSADA")
print("=" * 60)

print(
    f"\nParticipantes com matriz FC: "
    f"{len(df_final):,}"
)

print(
    f"ROIs: "
    f"{len(roi_ref)}"
)

print(
    f"Conexões únicas: "
    f"{config.N_CONNECTIONS:,}"
)

print("\nDistribuição das classes:")

display(
    df_final["classe"]
    .value_counts()
    .sort_index()
)

print("\nArquivos salvos:")

print(
    f"Participantes:\n"
    f"  {config.ARQ_PARTICIPANTES}"
)

print(
    f"\nROIs:\n"
    f"  {config.ARQ_ROIS}"
)

print(
    f"\nConectividades:\n"
    f"  {config.CONNECTIVITY}"
)

AMOSTRA FINAL PROCESSADA

Participantes com matriz FC: 346
ROIs: 190
Conexões únicas: 17,955

Distribuição das classes:


classe
0    185
1    161
Name: count, dtype: int64


Arquivos salvos:
Participantes:
  /home/maria/TCC_TDAH/data/processed/participants.csv

ROIs:
  /home/maria/TCC_TDAH/data/processed/roi_ids.npy

Conectividades:
  /home/maria/TCC_TDAH/data/connectivity
